# GIADA Task 33 — sinapsi osservabili e feedback autonomo
Protocollo v3 preregistrato dopo un arresto tecnico v1 e un floor v2 non ammissibile; nuovi seed e schedule, stessi gate numerici. Codice/notebook/config dalla stessa revisione Git. Run autonomo: nessun Input Kaggle esterno richiesto; non modifica i checkpoint congelati.


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
WORK = Path('/kaggle/working/giada_task33')
WORK.mkdir(parents=True, exist_ok=True)
REPO, TEACHER = WORK/'giada', WORK/'neuron_as_deep_net'
if not REPO.exists(): subprocess.run(['git','clone','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
if not TEACHER.exists(): subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
try: import neuron
except ImportError: subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True); import neuron
assert neuron.__version__.split('+')[0]=='8.2.7', neuron.__version__
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29','neuron':neuron.__version__})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_task33_observable_synaptic_feedback')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}; usa una sessione pulita.'
command=[sys.executable,str(REPO/'scripts/run_task33_observable_synaptic_feedback.py'),'--output',str(OUTPUT),'--teacher',str(TEACHER)]
completed=subprocess.run(command,cwd=REPO,check=False)
report_file=OUTPUT/'final_report.json' if (OUTPUT/'final_report.json').is_file() else OUTPUT/'failure_report.json'
report=json.loads(report_file.read_text()) if report_file.is_file() else {'valid':False,'diagnosis':'NO_REPORT'}
print({'process_exit':completed.returncode,'valid':report.get('valid'),'diagnosis':report.get('diagnosis'),'floor':report.get('native_floor_admissible'),'models_judged':report.get('model_judged'),'task33_passed':report.get('task33_passed'),'error':report.get('error')})
archive=shutil.make_archive('/kaggle/working/giada_task33_observable_synaptic_feedback','zip',root_dir=OUTPUT)
print({'archive':archive,'code_revision':REVISION})
